In [1]:
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


In [2]:
DATA_PATH = "../data/processed/clean_twitter_training.csv"
MODEL_PATH = "../models/sentiment_pipeline.pkl"

In [3]:

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["clean_tweet", "sentiment"])
df = df[df["clean_tweet"].astype(str).str.strip() != ""]


In [26]:
df["sentiment"] = df["sentiment"].replace({
    "Irrelevant": "Neutral"
})

neutral_examples = pd.DataFrame({
    "clean_tweet": [
        "the server maintenance will start today",
        "server maintenance will start at pm today",
        "the server maintenance is scheduled for today",
        "scheduled server maintenance will begin at pm",
        "the system maintenance will start tonight",
        "maintenance is planned for the server today",
        "the server update is scheduled for tonight",
        "the update was released today",
        "the meeting is scheduled for tomorrow",
        "the product launch event is scheduled for tomorrow",
        "the event will start at noon",
        "the application version has been updated",
        "the patch notes are available now",
        "the company announced a new update",
        "the service notice was published today"
    ],
    "sentiment": ["Neutral"] * 15
})

df = pd.concat([df, neutral_examples], ignore_index=True)

X = df["clean_tweet"].astype(str)
y = df["sentiment"].astype(str)

In [27]:
print(df["sentiment"].value_counts())

sentiment
Neutral     30805
Negative    22254
Positive    20580
Name: count, dtype: int64


In [28]:
X = df["clean_tweet"].astype(str)
y = df["sentiment"].astype(str)


In [29]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)






In [30]:

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ("model", LinearSVC(
        C=0.3,
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    )),
])

In [31]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [32]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8789380771319935

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.89      0.89      4451
     Neutral       0.89      0.88      0.88      6161
    Positive       0.87      0.86      0.87      4116

    accuracy                           0.88     14728
   macro avg       0.88      0.88      0.88     14728
weighted avg       0.88      0.88      0.88     14728


Confusion Matrix:
[[3972  325  154]
 [ 350 5414  397]
 [ 186  371 3559]]


In [33]:
joblib.dump(pipeline, MODEL_PATH)

print("\nModel saved successfully:", MODEL_PATH)


Model saved successfully: ../models/sentiment_pipeline.pkl


In [34]:
import joblib

loaded_model = joblib.load("../models/sentiment_pipeline.pkl")
print(loaded_model.classes_)

['Negative' 'Neutral' 'Positive']
